# Lab 07 — 06 Reconciliation



In [ ]:
from pathlib import Path
import sys
cwd=Path.cwd().resolve()
project_root=next((p for p in [cwd,*cwd.parents] if (p/'src'/'lab07').exists()),None)
if project_root and str(project_root/'src') not in sys.path: sys.path.insert(0,str(project_root/'src'))
if project_root and str(project_root/'tools') not in sys.path: sys.path.insert(0,str(project_root/'tools'))
dbutils.widgets.text('catalog','dbr_dev','01 Catalog'); dbutils.widgets.text('schema','parvinbadalov','02 Schema'); dbutils.widgets.text('volume_name','lab07_data_quality','03 Volume'); dbutils.widgets.text('run_id','manual','04 Run ID')
catalog=dbutils.widgets.get('catalog'); schema=dbutils.widgets.get('schema'); volume_name=dbutils.widgets.get('volume_name'); run_id=dbutils.widgets.get('run_id')
assert catalog=='dbr_dev' and schema=='parvinbadalov', f'Lab 07 requires dbr_dev.parvinbadalov, got {catalog}.{schema}'
volume_root=f'/Volumes/{catalog}/{schema}/{volume_name}'


In [ ]:
from pyspark.sql import functions as F
from lab07.reconciliation import reconcile_bronze,reconcile_gold
b=spark.table(f'{catalog}.{schema}.business_license_bronze').count(); v=spark.table(f'{catalog}.{schema}.business_license_validated').count(); q=spark.table(f'{catalog}.{schema}.business_license_quarantine').count(); g=int(spark.table(f'{catalog}.{schema}.license_quality_daily').agg(F.sum('trusted_rows')).first()[0] or 0)
r1=reconcile_bronze(b,v,q); r2=reconcile_gold(v,g); display(spark.createDataFrame([r1,r2])); assert r1['passed'] and r2['passed']; print('RECONCILIATION: PASS')
